In [ ]:
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import cv2

sys.path.append("..")
from tflite_inference import TFLiteInferer, TFLiteMerged

%load_ext autoreload
%autoreload 2

In [ ]:
def draw_output(img, output):
    tmp = img.copy()
    for box in output:
        x1, y1, w, h, conf, _, x, y = box
        x2 = x1 + w
        y2 = y1 + h
        cv2.rectangle(tmp, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.circle(tmp, (int(x), int(y)), 9, (0, 255, 0), -1)
        cv2.putText(tmp, f"{conf:.2f}", (int(x1), int(y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    plt.imshow(tmp)

In [ ]:
img_pth = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/images/val/4.2_batch2____4.2_H_cam_262_spacesMR_201.jpg"
img = cv2.imread(img_pth)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img)

In [ ]:
base_model = TFLiteInferer(
    yolo_pth="best_saved_model/best_full_integer_quant.tflite",
    head_pth="head_saved_model/head_float32.tflite",
    conf_th=0.20
)

In [ ]:
output = base_model.infer(img)
output

In [ ]:
draw_output(img, output)

In [ ]:
sep_model = TFLiteInferer(
    yolo_pth="best_saved_model/best_full_integer_quant.tflite",
    head_pth="head_uint8_saved_model/head_float32.tflite",
    conf_th=0.20,
    separated_head=True
)

In [ ]:
output = sep_model.infer(img)
output

In [ ]:
draw_output(img, output)

In [ ]:
sep_quant_model = TFLiteInferer(
    yolo_pth="best_saved_model/best_full_integer_quant.tflite",
    head_pth="head_uint8_saved_model_100/head_full_integer_quant.tflite",
    conf_th=0.35,
    separated_head=True,
    head_int8=True,
    iou_th=0.35,
    # idxs=[1, 6, 4, 3, 2, 5, 0]  # Adjust the indices based on your model's output structure
)

In [ ]:
output = sep_quant_model.infer(img)
output

In [ ]:
draw_output(img, output)

In [ ]:
sep_quant_model = TFLiteInferer(
    yolo_pth="best_saved_model/best_full_integer_quant.tflite",
    head_pth="head_uint8_saved_model_200/head_full_integer_quant.tflite",
    conf_th=0.35,
    separated_head=True,
    head_int8=True,
    iou_th=0.35,
    # idxs=[1, 6, 4, 3, 2, 5, 0]  # Adjust the indices based on your model's output structure
)

In [ ]:
output = sep_quant_model.infer(img)
output

In [ ]:
draw_output(img, output)

In [ ]:
sep_quant_model = TFLiteInferer(
    yolo_pth="best_saved_model/best_full_integer_quant.tflite",
    head_pth="head_uint8_saved_model_300/head_full_integer_quant.tflite",
    conf_th=0.35,
    separated_head=True,
    head_int8=True,
    iou_th=0.35,
    # idxs=[1, 6, 4, 3, 2, 5, 0]  # Adjust the indices based on your model's output structure
)

In [ ]:
output = sep_quant_model.infer(img)
output

In [ ]:
draw_output(img, output)

In [ ]:
sep_quant_model = TFLiteInferer(
    yolo_pth="best_saved_model/best_full_integer_quant.tflite",
    head_pth="head_uint8_saved_model_400/head_full_integer_quant.tflite",
    conf_th=0.35,
    separated_head=True,
    head_int8=True,
    iou_th=0.35,
    # idxs=[1, 6, 4, 3, 2, 5, 0]  # Adjust the indices based on your model's output structure
)

In [ ]:
output = sep_quant_model.infer(img)
output

In [ ]:
draw_output(img, output)

In [ ]:
import onnxruntime as ort
import numpy as np

In [ ]:
# Load the merged model
session = ort.InferenceSession("combined.onnx", providers=["CPUExecutionProvider"])

In [ ]:
input_name = session.get_inputs()[0].name
input_name

In [ ]:
input_data = sep_quant_model.pre_process(img)
input_data.shape
input_data = np.transpose(input_data, (0, 3, 1, 2)).astype(np.float32) / 255  # Ensure the input is in float32 format

In [ ]:
bboxes, scores = session.run(None, {input_name: input_data})

In [ ]:
outputs = np.concatenate([bboxes[:, :, :4], scores, bboxes[:, :, 4:]], axis=-1)
outputs.shape

In [ ]:
detections = sep_quant_model.post_process(outputs)
detections.shape

In [ ]:
draw_output(img, detections)

In [ ]:
combined_quant_model = TFLiteInferer(
    yolo_pth="combined/combined_full_integer_quant.tflite",
    conf_th=0.25,
    iou_th=0.35
)

In [ ]:
scores, bboxes = combined_quant_model.infer_no_head(img, quantized_output=False)
scores.shape, bboxes.shape

In [ ]:
raw_outputs = np.concatenate([bboxes[:, :, :4], scores, bboxes[:, :, 4:]], axis=-1)
outputs = combined_quant_model.post_process(raw_outputs)
outputs

In [ ]:
draw_output(img, outputs)

In [ ]:
outputs = combined_quant_model.infer_bottleneck(img)
outputs

In [ ]:
draw_output(img, outputs)

In [ ]:
combined_quant_model = TFLiteInferer(
    yolo_pth="combined_per_channel/combined_full_integer_quant.tflite",
    conf_th=0.25,
    iou_th=0.4
)

In [ ]:
outputs = combined_quant_model.infer_bottleneck(img)
outputs

In [ ]:
draw_output(img, outputs)

In [ ]:
full_quant_model = TFLiteInferer(
    yolo_pth="best_full_saved_model/best_full_full_integer_quant.tflite",
    conf_th=0.25,
)

In [ ]:
raw_output = full_quant_model.infer_no_head(img)[0]
raw_output.shape, raw_output[:, 4, :].max()

In [ ]:
outputs = full_quant_model.post_process(raw_output)
outputs

In [ ]:
combined_quant_model = TFLiteMerged(
    yolo_pth="combined_per_channel/combined_full_integer_quant.tflite",
    conf_th=0.25,
    iou_th=0.35
)

In [ ]:
outputs = combined_quant_model.infer(img)
outputs

In [ ]:
draw_output(img, outputs)

In [ ]:
combined_quant_model = TFLiteMerged(
    yolo_pth="best_saved_model/best_full_integer_quant.tflite",
    conf_th=0.25,
    iou_th=0.35
)

In [ ]:
outputs = combined_quant_model.infer(img)
outputs

In [ ]:
draw_output(img, outputs)